# 1.2.2 单期市场模型：问题与答案

## 题目背景

考虑两种资产，其当前现货价格分别为 30 美元和 50 美元。假设三个月后，第一种资产的价格可能为 34 美元或 24 美元，第二种资产的价格可能为 56 美元、51 美元或 46 美元。

以第一种资产为标的、执行价为 30 美元、三个月后到期的平值欧式看涨期权和看跌期权，当前价格分别为 2.6 美元和 2.3 美元。以第二种资产为标的、执行价为 50 美元、三个月后到期的平值欧式看涨期权和看跌期权，当前价格分别为 2.7 美元和 2.2 美元。

假设今天的 1 美元现金在三个月后的价值为 1.01 美元。

根据以上信息，可以构建一个期限为三个月、包含七种证券和六种状态的市场模型，具体如下。

**七种证券：**

1. 现金；
2. 第一种资产；
3. 第二种资产；
4. 以第一种资产为标的、执行价为 30 美元、三个月后到期的平值欧式看涨期权；
5. 以第一种资产为标的、执行价为 30 美元、三个月后到期的平值欧式看跌期权；
6. 以第二种资产为标的、执行价为 50 美元、三个月后到期的平值欧式看涨期权；
7. 以第二种资产为标的、执行价为 50 美元、三个月后到期的平值欧式看跌期权。

**三个月后市场的六种状态：**

| 状态 | 第一种资产的价格（美元） | 第二种资产的价格（美元） |
|---|---:|---:|
| $\omega^1$ | 34 | 56 |
| $\omega^2$ | 34 | 51 |
| $\omega^3$ | 34 | 46 |
| $\omega^4$ | 24 | 56 |
| $\omega^5$ | 24 | 51 |
| $\omega^6$ | 24 | 46 |

因此，这个市场模型包含 $m=7$ 种证券和 $n=6$ 种状态。


## 问题 1：用向量和数组表示题干中的价格与状态

按题干列出的七种证券顺序，当前价格列向量为

$$
S_0=(1,30,50,2.6,2.3,2.7,2.2)^T.\tag{1.54}
$$

资产 1 的未来价格集合为 $\{34,24\}$，资产 2 为 $\{56,51,46\}$。本模型将所有组合列为可能状态：

$$
\{34,24\}\times\{56,51,46\}
=\{(34,56),(34,51),(34,46),(24,56),(24,51),(24,46)\}.
$$

因此 $m=7$，$n=2\times3=6$。每个有序对表示同一到期日两种资产的价格（不一定等概率）。


In [27]:
import numpy as np
import pandas as pd
from itertools import product
from IPython.display import display

pd.set_option("display.float_format", "{:.4f}".format)

securities = ["现金", "资产1", "资产2", "资产1看涨", "资产1看跌", "资产2看涨", "资产2看跌"]
m = len(securities)
tau = 1 / 4  # 三个月；本例只考察今天和三个月后两个时点
cash_growth = 1.01
K1, K2 = 30.0, 50.0
S_0 = np.array([1.0, 30.0, 50.0, 2.6, 2.3, 2.7, 2.2])

display(pd.DataFrame({"当前单位价格": S_0}, index=securities))

# 定义六种未来状态；n 为状态数，供后续支付矩阵计算使用。
state_prices = np.array(list(product([34.0, 24.0], [56.0, 51.0, 46.0])))
n = len(state_prices)
states = [f"ω{k}" for k in range(1, n + 1)]
asset1_tau = state_prices[:, 0]
asset2_tau = state_prices[:, 1]

display(pd.DataFrame(state_prices, index=states, columns=["资产1到期价格", "资产2到期价格"]))
print(f"初始化完成：{m} 种证券，{n} 种状态。")

,当前单位价格
现金,1.0000
资产1,30.0000
资产2,50.0000
资产1看涨,2.6000
资产1看跌,2.3000
资产2看涨,2.7000
资产2看跌,2.2000


,资产1到期价格,资产2到期价格
ω1,34.0000,56.0000
ω2,34.0000,51.0000
ω3,34.0000,46.0000
ω4,24.0000,56.0000
ω5,24.0000,51.0000
ω6,24.0000,46.0000


初始化完成：7 种证券，6 种状态。


## 问题 2：每种证券到期值多少？建立支付矩阵

题目已经给了两种资产的未来价格；现金和四种期权在这些状态下的价值，需要据此算出。把每种证券的六个结果排成一行，就能建立支付矩阵。

### 2.1 计算每种证券的到期价值


看涨期权允许持有人按执行价 $K$ 买入资产。当 $S(\tau)>K$ 时，这个权利值 $S(\tau)-K$；否则可以放弃，价值为零：

$$
C(\tau)=\max(S(\tau)-K,0)
=\begin{cases}S(\tau)-K,&S(\tau)>K,\\0,&S(\tau)\le K.\end{cases}\tag{1.58}
$$

看跌期权允许持有人按 $K$ 卖出资产，因此

$$
P(\tau)=\max(K-S(\tau),0)
=\begin{cases}0,&S(\tau)\ge K,\\K-S(\tau),&S(\tau)<K.\end{cases}\tag{1.59}
$$

以资产 1 的看涨期权为例：今天花 2.6 美元买入，三个月后有权以 30 美元买入资产 1。

| 到期时资产 1 的价格 | 是否行使买入权利 | 期权到期价值 | 扣除 2.6 美元买入成本后的盈亏 |
|---|---|---:|---:|
| 34 美元 | 行使：花 30 美元买到价值 34 美元的资产 | $34-30=4$ 美元 | $4-2.6=1.4$ 美元 |
| 24 美元 | 放弃：市场上只值 24 美元，无需花 30 美元买 | 0 美元 | $0-2.6=-2.6$ 美元 |

本题支付矩阵要填的是“期权到期价值”，所以这一行填 4 或 0。买期权花的 2.6 美元用于计算个人盈亏，不从支付矩阵中扣除。表中的盈亏暂不考虑利息和交易费用。

**补充理解：买入与做空期权。**

先分清两个维度：看涨／看跌是合约类型，买入／卖出是持仓方向，组合起来有四种头寸。这里的“卖出”专指卖出开仓、承担期权义务，也叫做空期权，不是卖掉自己原来持有的期权。

以下沿用资产 1 的条件：执行价 $K=30$ 美元，看涨期权价格为 2.6 美元，看跌期权价格为 2.3 美元，每份对应一单位资产。

| 头寸 | 买到的权利或承担的义务 | 今天的现金收支 | 到期资产涨到 34 美元 | 到期资产跌到 24 美元 |
|---|---|---|---|---|
| **买入看涨期权（long call）** | 有权按 30 美元买入资产；可以放弃行权 | - 2.6 美元 | 权利值 $34-30=4$ 美元；净赚 $4-2.6=1.4$ 美元 | 放弃行权，期权值 0；净亏 2.6 美元 |
| **做空看涨期权（short call）** | 对方行权时，有义务按 30 美元卖出资产 | + 2.6 美元 | 承担 $34-30=4$ 美元的到期支付；净亏 $4-2.6=1.4$ 美元 | 对方不行权，到期无需支付；净赚 2.6 美元 |
| **买入看跌期权（long put）** | 有权按 30 美元卖出资产；可以放弃行权 | - 2.3 美元 | 放弃行权，期权值 0；净亏 2.3 美元 | 权利值 $30-24=6$ 美元；净赚 $6-2.3=3.7$ 美元 |
| **做空看跌期权（short put）** | 对方行权时，有义务按 30 美元买入资产 | + 2.3 美元 | 对方不行权，到期无需支付；净赚 2.3 美元 | 承担 $30-24=6$ 美元的到期支付；净亏 $6-2.3=3.7$ 美元 |

上表盈亏忽略利息和交易费用。后面计算组合时，做空通过负持仓表示；支付矩阵本身仍记录每种证券一单位多头的到期价值。

将两个标的各自的未来价格代入，四种期权在六种状态下的结果依次为

$$
\begin{aligned}
C_1(\tau)&:\ (4,4,4,0,0,0),\\
P_1(\tau)&:\ (0,0,0,6,6,6),\\
C_2(\tau)&:\ (6,1,0,6,1,0),\\
P_2(\tau)&:\ (0,0,4,0,0,4).
\end{aligned}
$$

现金在所有状态下都为 1.01。下面将到期价值公式逐元素应用于六种状态。

符号中，$C_1$、$P_1$ 的下标 1 指资产 1，$C_2$、$P_2$ 的下标 2 指资产 2；状态用 $\omega^k$ 区分。下文用 $A_1(\tau)$、$A_2(\tau)$ 分别表示两种标的资产的到期价格，避免与七种证券的编号混淆。

In [28]:
cash_tau = np.full(n, cash_growth)
call1_tau = np.maximum(asset1_tau - K1, 0)
put1_tau = np.maximum(K1 - asset1_tau, 0)
call2_tau = np.maximum(asset2_tau - K2, 0)
put2_tau = np.maximum(K2 - asset2_tau, 0)

for name, values in [
    ("资产1看涨", call1_tau), ("资产1看跌", put1_tau),
    ("资产2看涨", call2_tau), ("资产2看跌", put2_tau),
]:
    print(f"{name}：{values}")

资产1看涨：[4. 4. 4. 0. 0. 0.]
资产1看跌：[0. 0. 0. 6. 6. 6.]
资产2看涨：[6. 1. 0. 6. 1. 0.]
资产2看跌：[0. 0. 4. 0. 0. 4.]


### 2.2 将各证券的结果逐行堆叠，得到支付矩阵

这里 $j$ 按题干中七种证券的顺序编号：1 是现金，2 是资产 1，3 是资产 2，后四项是期权。令 $S_{j,\tau}$ 是第 $j$ 种证券在六种状态下的价格构成的列向量，那么矩阵的第 $j$ 行就是 $(S_{j,\tau})^T$。依次堆叠：

$$
M_{\tau}=\begin{bmatrix}
(S_{1,\tau})^T\\(S_{2,\tau})^T\\\vdots\\(S_{7,\tau})^T
\end{bmatrix}
=\begin{pmatrix}
1.01&1.01&1.01&1.01&1.01&1.01\\
34&34&34&24&24&24\\
56&51&46&56&51&46\\
4&4&4&0&0&0\\
0&0&0&6&6&6\\
6&1&0&6&1&0\\
0&0&4&0&0&4
\end{pmatrix}.\tag{1.64}
$$

同一个矩阵也能按列读取：

$$M_{\tau}=\begin{bmatrix}S_{\tau}^{\omega^1}&\cdots&S_{\tau}^{\omega^6}\end{bmatrix}.\tag{1.65}$$


In [29]:
M_tau = np.vstack([cash_tau, asset1_tau, asset2_tau, call1_tau, put1_tau, call2_tau, put2_tau])
payoff_table = pd.DataFrame(M_tau, index=securities, columns=states)
display(payoff_table)

,ω1,ω2,ω3,ω4,ω5,ω6
现金,1.0100,1.0100,1.0100,1.0100,1.0100,1.0100
资产1,34.0000,34.0000,34.0000,24.0000,24.0000,24.0000
资产2,56.0000,51.0000,46.0000,56.0000,51.0000,46.0000
资产1看涨,4.0000,4.0000,4.0000,0.0000,0.0000,0.0000
资产1看跌,0.0000,0.0000,0.0000,6.0000,6.0000,6.0000
资产2看涨,6.0000,1.0000,0.0000,6.0000,1.0000,0.0000
资产2看跌,0.0000,0.0000,4.0000,0.0000,0.0000,4.0000


这就是所求的 $7\times6$ 支付矩阵。

- 第 $j$ 行：第 $j$ 种证券在共 6 种状态下值多少钱。
- 第 $k$ 列：第 $k$ 种状态下，共 7 种证券分别值多少钱。

例如，资产 1 看涨期权这一行是 `[4, 4, 4, 0, 0, 0]`；状态 $\omega^2$ 这一列是 `[1.01, 34, 51, 4, 0, 1, 0]`。

## 问题 3：给定以下持仓，今天和三个月后的组合价值分别是多少？

支付矩阵已经给出每种证券的到期价值。现在给定一个具体组合，把各证券的价值按持仓数量相加。以下是建立组合后的持仓：

- 持有 10,000 美元现金；
- 买入资产 1 共 200 单位；
- 做空资产 2 共 100 单位；
- 卖空资产 1 看涨期权 1,000 份；
- 持仓资产 1 看跌期权为零；
- 买入资产 2 看涨期权 500 份；
- 买入资产 2 看跌期权 600 份。


假设三个月内持仓不变。通过本例我们可以看清持仓向量怎样把七种证券的价值加总成一个组合的价值。

正数表示做多，负数表示做空，持仓向量为

$$\Theta=(10000,200,-100,-1000,0,500,600)^T.$$

下面按原书的推导顺序，先算当前价值，再算某一种未来状态下的价值，最后合并全部状态。沿用笔记的约定：向量默认是列向量。一般公式用 $t_0$ 和 $\tau$ 表示当前与未来时刻；本例 $t_0=0$，$\tau=1/4$ 年。

### 3.1 当前组合价值


本例假设持仓数量不变，从今天到三个月后不再买卖，所以设第 $j$ 种证券持有 $\Theta_j$ 单位，当前单价为 $S_{j,t_0}$。每项持仓的价值是 $\Theta_j S_{j,t_0}$，全部相加：

$$
V_{t_0}
=\Theta_1S_{1,t_0}+\Theta_2S_{2,t_0}+\cdots+\Theta_mS_{m,t_0}
=\sum_{j=1}^{m}\Theta_j S_{j,t_0}.
$$

令 $\Theta=(\Theta_1,\ldots,\Theta_m)^T$，$S_{t_0}=(S_{1,t_0},\ldots,S_{m,t_0})^T$，由行向量乘列向量的定义，上式就是

$$
V_{t_0}=\Theta^T S_{t_0}.
$$

维度为 $(1\times m)(m\times1)=1\times1$，所以结果是一个数。

In [30]:
theta = np.array([10000.0, 200.0, -100.0, -1000.0, 0.0, 500.0, 600.0])
V_0 = theta @ S_0
print(f"当前组合价值：{V_0:,.2f} 美元")

当前组合价值：11,070.00 美元


### 3.2 一个未来状态下的组合价值

如果三个月后恰好出现状态 $\omega^k$，组合值多少钱？

假设持仓不变，只把当前价格换成状态 $\omega^k$ 下的价格：

$$
\begin{aligned}
V_{\tau}^{\omega^k}
&=\Theta_1S_{1,\tau}^{\omega^k}+\cdots+\Theta_mS_{m,\tau}^{\omega^k}\\
&=\sum_{j=1}^{m}\Theta_jS_{j,\tau}^{\omega^k}\\
&=\Theta^T S_{\tau}^{\omega^k}.
\end{aligned}\tag{1.67}
$$

这里 $S_{\tau}^{\omega^k}$ 就是支付矩阵 $M_\tau$ 的第 $k$ 列。下面取状态 $\omega^2$，即资产 1 值 34 美元、资产 2 值 51 美元，演示如何取出一列并计算。Python 下标从 0 开始，因此取 `M_tau[:, 1]`。

In [31]:
k = 1  # Python 下标 1 对应原书状态 ω2。
state_prices_k = M_tau[:, k]
V_state = theta @ state_prices_k

display(pd.DataFrame({
    "持仓单位": theta,
    f"{states[k]} 到期单位价值": state_prices_k,
    "对组合到期价值的贡献": theta * state_prices_k,
}, index=securities))
print(f"若出现状态 {states[k]}，组合到期值 {V_state:,.2f} 美元。")

,持仓单位,ω2 到期单位价值,对组合到期价值的贡献
现金,10000.0000,1.0100,10100.0000
资产1,200.0000,34.0000,6800.0000
资产2,-100.0000,51.0000,-5100.0000
资产1看涨,-1000.0000,4.0000,-4000.0000
资产1看跌,0.0000,0.0000,0.0000
资产2看涨,500.0000,1.0000,500.0000
资产2看跌,600.0000,0.0000,0.0000


若出现状态 ω2，组合到期值 8,300.00 美元。


### 3.3 全部未来状态下的组合支付向量

怎样一次算出全部 $n$ 种状态下的组合价值：把六个内积合成一次矩阵乘法。

先把各状态下的组合价值排成行向量：

$$
(V_\tau)^T
=\begin{pmatrix}V_\tau^{\omega^1}&\cdots&V_\tau^{\omega^n}\end{pmatrix}.
\tag{1.66}
$$

由上一步，每个元素都可以写成 $\Theta^T S_\tau^{\omega^k}$。把它们放在一起，提取共同的左乘向量 $\Theta^T$：

$$
\begin{aligned}
(V_\tau)^T
&=\begin{pmatrix}\Theta^T S_\tau^{\omega^1}&\cdots&\Theta^T S_\tau^{\omega^n}\end{pmatrix}\\
&=\Theta^T\begin{bmatrix}S_\tau^{\omega^1}&\cdots&S_\tau^{\omega^n}\end{bmatrix}\\
&=\Theta^T M_\tau.
\end{aligned}\tag{1.68}
$$

最后一步使用支付矩阵的列形式：其第 $k$ 列就是状态 $\omega^k$ 下所有证券的价格。

维度为 $(1\times m)(m\times n)=1\times n$。如果要把结果存为列向量，也可以转置为 $V_\tau=M_\tau^T\Theta$。代码中 `theta` 使用一维数组，直接用 `theta @ M_tau` 得到六个结果，再放进表格。

In [32]:
V_tau = theta @ M_tau

display(pd.DataFrame({
    "组合到期价值": V_tau,
    "相对初始价值的变动": V_tau - V_0,
}, index=states))

,组合到期价值,相对初始价值的变动
ω1,10300.0000,-770.0000
ω2,8300.0000,-2770.0000
ω3,10700.0000,-370.0000
ω4,12300.0000,1230.0000
ω5,10300.0000,-770.0000
ω6,12700.0000,1630.0000


今天组合值 **11,070 美元**；六种状态下的到期价值依次为 **10,300；8,300；10,700；12,300；10,300；12,700 美元**。

这里 `theta` 记录“七种证券各持有多少”，`V_tau` 记录“组合在六种状态下各值多少”。给定持仓求支付，就是 `theta @ M_tau`。下一步反过来：给定想要的支付，寻找持仓。


## 问题 4：指定看跌期权的支付，能否用其他证券复制？

问题 3 已知持仓，计算支付；现在已知目标支付，寻找持仓。以资产 1 看跌期权为例，目标是六个状态下的 $(0,0,0,6,6,6)$。我们要求看跌期权本身的持仓为零，用其他证券拼出这六个数。

这叫作**复制**：在每一种状态下，组合的到期价值都与目标证券相同。这里比较的是到期价值；建立组合的当前成本在拓展问题 6 中讨论。

复制组合的支付向量，是已有证券支付向量的线性组合。组合中每种证券的持仓数量，就是对应支付向量的系数。令 $s_\tau$ 是要复制的证券在全部状态下的价值列向量，则

$$
(s_\tau)^T=(V_\tau)^T=\Theta^T M_\tau
=\sum_{j=1}^{m}\Theta_j(S_{j,\tau})^T.\tag{1.69}
$$

对于复制问题，$s_\tau$ 和 $M_\tau$ 已知，需要寻找 $\Theta$。本例可以直接利用期权支付之间的关系构造持仓。

### 4.1 寻找关系：看涨、看跌与标的资产的到期价值

取同一标的、同执行价 $K$、同到期日的看涨和看跌期权。分两种情况代入它们的到期价值：

$$
P(\tau)+S(\tau)-C(\tau)=
\begin{cases}
0+S(\tau)-(S(\tau)-K)=K,&S(\tau)\ge K,\\
(K-S(\tau))+S(\tau)-0=K,&S(\tau)<K.
\end{cases}
$$

所以无论资产价格是多少，总有

$$P(\tau)+S(\tau)-C(\tau)=K.\tag{1.72}$$


在 $P(\tau)+S(\tau)-C(\tau)$ 中，$P$ 表示持有一份看跌期权，$S$ 表示持有一单位资产，$-C$ 表示卖出一份看涨期权。以 $K=30$ 为例：

| 到期资产价格 | 持有看跌期权的价值 $P(\tau)$ | 持有资产的价值 $S(\tau)$ | 看涨期权空头的支付 $-C(\tau)$ | 合计 |
|---|---:|---:|---:|---:|
| 34 美元 | 0 | 34 | $-4$ | 30 |
| 24 美元 | 6 | 24 | 0 | 30 |

如果按实物交割理解：涨到 34 时，看涨买方可以按 30 美元买走你持有的资产；跌到 24 时，你可以行使看跌期权，将手里的资产按 30 美元卖给看跌合约的义务方。两种情况下组合最后都换回 30 美元。

接下来将六个状态分别代入，检查结果是不是始终等于执行价。

In [33]:
parity1_tau = put1_tau + asset1_tau - call1_tau
parity2_tau = put2_tau + asset2_tau - call2_tau

display(pd.DataFrame({
    "资产1：P + S - C": parity1_tau,
    "资产2：P + S - C": parity2_tau,
}, index=states))

,资产1：P + S - C,资产2：P + S - C
ω1,30.0000,50.0000
ω2,30.0000,50.0000
ω3,30.0000,50.0000
ω4,30.0000,50.0000
ω5,30.0000,50.0000
ω6,30.0000,50.0000


### 4.2 资产1看跌期权的复制

**不买资产 1 的看跌期权，能不能用其他证券组成一个组合，在每一种状态下，都得到与这份看跌期权相同的到期价值？**

这里的 $P_1(\tau)$ 表示资产 1 看跌期权的到期价值：资产涨到 34 美元时，它值 0；跌到 24 美元时，它值 6 美元。我们要找的组合，也必须在这两种情况下分别值 0 和 6 美元。这叫作复制看跌期权。

由 $P_1(\tau)+A_1(\tau)-C_1(\tau)=30$，移项得

$$P_1(\tau)=30-A_1(\tau)+C_1(\tau).$$

这三个加减号分别对应：准备到期值 30 美元的现金、做空一单位资产 1、买入一份资产 1 的看涨期权。到期时：

| 资产 1 的到期价格 | 现金价值 | 归还所欠资产的负债价值 | 看涨期权价值 | 组合合计 | 看跌期权价值 $P_1(\tau)$ |
|---|---:|---:|---:|---:|---:|
| 34 美元 | 30 | −34 | 4 | **0** | **0** |
| 24 美元 | 30 | −24 | 0 | **6** | **6** |

两行都与看跌期权一致，因此这个组合可以替代看跌期权的到期结果；同时今天一单位现金到期变成 1.01 美元，所以要得到确定的 30 美元，今天需持有 $30/1.01$ 单位现金。

按六种状态写成行向量，编号 1、2、4、5 分别是现金、资产 1、资产 1 看涨、资产 1 看跌，原书关系为：

$$
(S_{5,\tau})^T+(S_{2,\tau})^T-(S_{4,\tau})^T
=(30,30,30,30,30,30)
=\frac{30}{1.01}(S_{1,\tau})^T.\tag{1.70}
$$

移项后

$$
(S_{5,\tau})^T
=\frac{30}{1.01}(S_{1,\tau})^T-(S_{2,\tau})^T+(S_{4,\tau})^T.
$$

复制持仓为

$$\Theta^{(P_1)}=(30/1.01,-1,0,1,0,0,0)^T.$$


In [34]:
theta_replicate_put1 = np.array([K1 / cash_growth, -1.0, 0.0, 1.0, 0.0, 0.0, 0.0])
replicated_put1_tau = theta_replicate_put1 @ M_tau

np.testing.assert_allclose(replicated_put1_tau, put1_tau, atol=1e-12)

display(pd.DataFrame({
    "资产1看跌价值": put1_tau,
    "复制组合价值": replicated_put1_tau,
    "差额": replicated_put1_tau - put1_tau,
}, index=states))

,资产1看跌价值,复制组合价值,差额
ω1,0.0000,0.0000,0.0000
ω2,0.0000,0.0000,0.0000
ω3,0.0000,0.0000,0.0000
ω4,6.0000,6.0000,0.0000
ω5,6.0000,6.0000,0.0000
ω6,6.0000,6.0000,0.0000


### 4.3 资产 2 看跌期权的复制

**不买资产 2 的看跌期权，能否也用现金、资产 2 和它的看涨期权，复制这份看跌期权的到期结果？**

这一节的目标是 $P_2(\tau)$，即资产 2 看跌期权的到期价值。资产 2 到期价格为 56、51、46 美元时，看跌期权分别值 0、0、4 美元，所以复制组合也必须逐一得到这三个数。

同理，$P_2(\tau)=50-A_2(\tau)+C_2(\tau)$。编号 1、3、6、7 分别是现金、资产 2、资产 2 看涨、资产 2 看跌。用证券编号写出原书关系：

$$
(S_{7,\tau})^T+(S_{3,\tau})^T-(S_{6,\tau})^T
=(50,50,50,50,50,50)
=\frac{50}{1.01}(S_{1,\tau})^T.\tag{1.71}
$$

移项得

$$
(S_{7,\tau})^T
=\frac{50}{1.01}(S_{1,\tau})^T-(S_{3,\tau})^T+(S_{6,\tau})^T.
$$

具体持仓是：准备到期值 50 美元的现金、做空一单位资产 2、买入一份资产 2 的看涨期权。三种价格下的组合价值分别为

$$
50-56+6=0,\qquad 50-51+1=0,\qquad 50-46+0=4.
$$

它们恰好等于看跌期权的到期价值，所以复制持仓为

$$\Theta^{(P_2)}=(50/1.01,0,-1,0,0,1,0)^T.$$

两种看跌期权的支付向量都可以由其余证券的支付向量线性组合得到，所以它们都是冗余证券。


In [35]:
theta_replicate_put2 = np.array([K2 / cash_growth, 0.0, -1.0, 0.0, 0.0, 1.0, 0.0])
replicated_put2_tau = theta_replicate_put2 @ M_tau

np.testing.assert_allclose(replicated_put2_tau, put2_tau, atol=1e-12)

display(pd.DataFrame({
    "资产2看跌价值": put2_tau,
    "复制组合价值": replicated_put2_tau,
    "差额": replicated_put2_tau - put2_tau,
}, index=states))

,资产2看跌价值,复制组合价值,差额
ω1,0.0000,0.0000,0.0000
ω2,0.0000,0.0000,0.0000
ω3,4.0000,4.0000,0.0000
ω4,0.0000,0.0000,0.0000
ω5,0.0000,0.0000,0.0000
ω6,4.0000,4.0000,0.0000


## 问题 5：七种证券提供了多少个独立的支付方向？

问题 4 已发现，两种看跌期权的支付可以由其他证券复制，因此七行并非彼此独立。接下来问：最少需要多少个独立支付行向量，才能生成全部七行？这个数就是支付矩阵的秩。

### 5.1 先计算秩：七行中有多少个独立方向？

原书先把“证券能否被其他证券复制”与行向量的线性相关联系起来：

记 $R_j=(S_{j,\tau})^T$ 为支付矩阵第 $j$ 行。若这些行向量线性相关，则存在不全为零的系数，使

$$\sum_{j=1}^{m}c_jR_j=0.$$

选一个 $c_\ell\ne0$，移项并除以 $c_\ell$：

$$R_\ell=-\sum_{j\ne\ell}\frac{c_j}{c_\ell}R_j.$$

这说明第 $\ell$ 种证券可由其余证券复制。反过来，有这种复制关系就能移项得到非零系数的线性关系。因此，原书的判别结论是

$$\text{没有冗余证券}\quad\Longleftrightarrow\quad\operatorname{rank}(M_{\tau})=m.$$

本例 $M_{\tau}$ 有 7 行、6 列，秩最多为 6，已经能确定存在冗余。


In [10]:
rank_M = np.linalg.matrix_rank(M_tau)
print(f"答案：支付矩阵有 {m} 行，秩为 {rank_M}，因此存在冗余证券。")

答案：支付矩阵有 7 行，秩为 4，因此存在冗余证券。


### 5.2 为什么去掉两种看跌期权后，剩下五种证券还只有四个独立方向？

函数给出的秩是 4，但去掉两种看跌期权后还剩五行。这说明其中仍有线性依赖。我们检查资产 1 的看涨期权：它能否只用现金和资产 1 复制？下面是针对本例的补充推导。

设持有 $\Delta$ 单位资产 1 和 $b$ 单位现金，希望在资产值 34 时得到 4 美元、值 24 时得到 0。于是

$$
\begin{cases}
34\Delta+1.01b=4,\\
24\Delta+1.01b=0.
\end{cases}
$$

两式相减得 $10\Delta=4$，所以 $\Delta=0.4$。代回第二式：

$$1.01b=-24\times0.4=-9.6,\qquad b=-9.6/1.01.$$

因此在全部六种状态下，

$$C_1(\tau)=0.4A_1(\tau)-9.6.$$

即 $\text{资产 1 看涨期权的到期价值}=0.4\times\text{资产 1 的到期价格}-9.6\text{ 美元}$。其中到期需偿还 9.6 美元，对应今天现金持仓 $b=-9.6/1.01$；负现金持仓表示借款。

这个等式只针对本例资产 1 到期价为 34 或 24 的设定。下面先把求出的现金、资产持仓放入七项持仓向量 `theta_replicate_call1`，再乘 `M_tau`，与目标 `call1_tau` 比较。


In [11]:
delta1 = (4.0 - 0.0) / (34.0 - 24.0)
cash_units1 = -delta1 * 24.0 / cash_growth
theta_replicate_call1 = np.array([cash_units1, delta1, 0.0, 0.0, 0.0, 0.0, 0.0]) # 复制资产 1 看涨期权所需的持仓向量
replicated_call1_tau = theta_replicate_call1 @ M_tau # 复制组合的支付向量 = 复制资产 1 看涨期权所需的持仓向量 x 支付矩阵

np.testing.assert_allclose(replicated_call1_tau, call1_tau, atol=1e-12)

print(f"复制资产1看涨期权：现金： {cash_units1:.2f} 单位，资产1： {delta1:.1f} 单位。")
display(pd.DataFrame({
    "资产1看涨价值": call1_tau,
    "复制组合价值": replicated_call1_tau,
}, index=states))

复制资产1看涨期权：现金： -9.50 单位，资产1： 0.4 单位。


,资产1看涨价值,复制组合价值
ω1,4.0000,4.0000
ω2,4.0000,4.0000
ω3,4.0000,4.0000
ω4,0.0000,-0.0000
ω5,0.0000,-0.0000
ω6,0.0000,-0.0000


### 5.3 根据复制关系，选出并验证四个独立行

前面在 5.1 已用 `np.linalg.matrix_rank(M_tau)` 算得 $\operatorname{rank}(M_\tau)=4$。现在根据已经推导出的复制关系，选择一组独立行，解释这个计算结果。

问题 4 已说明两种看跌期权可由其余证券复制；5.2 又说明资产 1 看涨期权可由现金和资产 1 复制。因此，选择剩下的四行作为候选：现金、资产 1、资产 2、资产 2 看涨期权。它们在 Python 中的行索引是 `[0, 1, 2, 5]`。



In [36]:
# 根据前面的复制关系，手动选择四行作为候选，再验证线性无关。
independent_rows = [0, 1, 2, 5]  # 现金、资产1、资产2、资产2看涨
basis_payoffs = M_tau[independent_rows, :]
assert np.linalg.matrix_rank(basis_payoffs) == len(independent_rows)
print("一组独立支付方向：", [securities[i] for i in independent_rows])
display(payoff_table.iloc[independent_rows])


一组独立支付方向： ['现金', '资产1', '资产2', '资产2看涨']


,ω1,ω2,ω3,ω4,ω5,ω6
现金,1.0100,1.0100,1.0100,1.0100,1.0100,1.0100
资产1,34.0000,34.0000,34.0000,24.0000,24.0000,24.0000
资产2,56.0000,51.0000,46.0000,56.0000,51.0000,46.0000
资产2看涨,6.0000,1.0000,0.0000,6.0000,1.0000,0.0000


**用这四行表示其余三行。**

取现金、资产 1、资产 2、资产 2 看涨期权的支付行为 $B$ 的四行。由前面的三个复制关系：

$$
\begin{aligned}
(S_{4,\tau})^T&=-\frac{9.6}{1.01}(S_{1,\tau})^T+0.4(S_{2,\tau})^T,\\
(S_{5,\tau})^T&=\frac{30-9.6}{1.01}(S_{1,\tau})^T-0.6(S_{2,\tau})^T,\\
(S_{7,\tau})^T&=\frac{50}{1.01}(S_{1,\tau})^T-(S_{3,\tau})^T+(S_{6,\tau})^T.
\end{aligned}
$$

上述关系表明，全部七行都在 $B$ 的四行生成的空间中。结合这四行线性无关，就解释了前面用函数算出的“秩为 4”：它们恰好提供四个独立的支付方向，其余行都能由它们组合得到。


### 5.4 补充验证：把复制关系集中写成一个系数矩阵

到 5.3，已经解释了为什么秩是 4。本节把前面的复制关系合在一起，验证四行能还原整个支付矩阵。

记 $B$ 为 `basis_payoffs`，它的四行依次是现金、资产 1、资产 2、资产 2 看涨期权的支付。构造系数矩阵 $L$（代码中的 `row_coefficients`）：

- **七行**对应原来的七种证券，每行说明如何复制该证券。
- **四列**对应选出的四种证券，每个元素是复制时的持仓数量。

例如第 4 行 `[-9.6 / cash_growth, 0.4, 0, 0]` 表示：借入现金、持有 0.4 单位资产 1，得到资产 1 看涨期权的支付。现金、资产 1、资产 2、资产 2 看涨期权各复制自己，其系数分别是四个单位行向量。

因此，每行系数乘 $B$，就得到一种证券的六项支付；把七行一起计算，就是

$$
\underbrace{L}_{7\times4}\underbrace{B}_{4\times6}
=\underbrace{M_\tau}_{7\times6}.
$$


In [37]:
row_coefficients = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [-9.6 / cash_growth, 0.4, 0.0, 0.0],
    [(K1 - 9.6) / cash_growth, -0.6, 0.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
    [K2 / cash_growth, 0.0, -1.0, 1.0],
])
display(pd.DataFrame(
    row_coefficients,
    index=securities,
    columns=[securities[i] for i in independent_rows],
))
reconstructed_M_tau = row_coefficients @ basis_payoffs # 用那四个独立行，重新拼出的完整支付矩阵，应当与原来的 M_tau 相同
np.testing.assert_allclose(reconstructed_M_tau, M_tau, atol=1e-12)
print("重建结果与原支付矩阵 M_tau 一致：四行足以还原全部七行。")

,现金,资产1,资产2,资产2看涨
现金,1.0000,0.0000,0.0000,0.0000
资产1,0.0000,1.0000,0.0000,0.0000
资产2,0.0000,0.0000,1.0000,0.0000
资产1看涨,-9.5050,0.4000,0.0000,0.0000
资产1看跌,20.1980,-0.6000,0.0000,0.0000
资产2看涨,0.0000,0.0000,0.0000,1.0000
资产2看跌,49.5050,0.0000,-1.0000,1.0000


重建结果与原支付矩阵 M_tau 一致：四行足以还原全部七行。
